# Session 20: Combining Tables

Data Science with Python, Level 1, Stage 2: NumPy and pandas

## Learning Objectives

By the end of this session you will be able to:

1. Explain why data is split into several tables.
2. Join tables on a key with `merge`.
3. Choose between inner, left and outer joins.
4. Find rows that do not match.
5. Stack tables with `concat`.
6. Reshape between wide and long formats.

## Where We Are

So far each analysis used one table. Real systems store data in **several tables** that refer to each other. An online shop keeps customers in one table, products in another and orders in a third. Each order only stores the **IDs** of its customer and product, not their names, cities or prices.

This avoids repeating the same customer details in every order, but it means that most questions need information from more than one table. Today we join them. The same idea is called a JOIN in SQL databases, which you will meet in many data jobs.

## Setup

Run this cell first. It loads the three tables of a fictional online shop that sells books, electronics, stationery and bags.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
customers = pd.read_csv(DATA_URL + "shop_customers.csv")
products = pd.read_csv(DATA_URL + "shop_products.csv")
orders = pd.read_csv(DATA_URL + "shop_orders.csv")

## 1. Tables and Keys

In [ ]:
print(customers.head(3))
print(products.head(3))
print(orders.head(3))

## 2. Joining with merge

In [ ]:
order_lines = orders.merge(products, on="product_id")

order_lines["total"] = order_lines["price"] * order_lines["quantity"]
order_lines[["product", "category", "price", "quantity", "total"]].head(4)

In [ ]:
full = order_lines.merge(customers, on="customer_id")
print(full.shape)

full.groupby(["city", "category"])["total"].sum().unstack().round(0)

## 3. Types of Join

What should happen to a row with no match in the other table? The `how` argument decides:

- `inner` (the default) keeps only rows whose key appears in **both** tables.
- `left` keeps **all** rows of the left table and fills missing details with `NaN`.
- `right` is the mirror image; `outer` keeps all rows from both tables.

Choosing the wrong join silently adds or drops rows, so always check the number of rows before and after.

In [ ]:
inner = orders.merge(customers, on="customer_id", how="inner")
left = orders.merge(customers, on="customer_id", how="left")

print(len(orders), len(inner), len(left))
print(left["name"].isna().sum(), "orders with no matching customer")

| how= | Keeps | Use it when |
| --- | --- | --- |
| `"inner"` | Only keys found in both tables | You only want complete records |
| `"left"` | Every row of the left table | Every order must stay, matched or not |
| `"right"` | Every row of the right table | The mirror of left; rarely needed |
| `"outer"` | Every row of both tables | You want to see all mismatches |

## 4. Finding Rows That Do Not Match

In [ ]:
check = orders.merge(customers, on="customer_id", how="left", indicator=True)
print(check["_merge"].value_counts())

unknown = check.loc[check["_merge"] == "left_only", "customer_id"]
print(sorted(unknown.unique()))

In [ ]:
no_orders = customers[~customers["customer_id"].isin(orders["customer_id"])]
print(len(no_orders), "customers never ordered")
no_orders[["customer_id", "name", "segment"]]

## 5. Stacking Tables with concat

`merge` puts tables **side by side** by matching keys. `pd.concat` puts tables **one under the other**, for example sales files from different months or branches with the same columns.

In [ ]:
jan = orders[orders["order_date"] < "2025-02-01"]
feb = orders[orders["order_date"].between("2025-02-01", "2025-02-28")]

both = pd.concat([jan, feb], ignore_index=True)
print(len(jan), len(feb), len(both))
both.tail(3)

## 6. Wide and Long Tables

The same data can be stored **wide**, with one column per month, or **long**, with one row per month. People read wide tables more easily; charts and `groupby` usually need long tables. `melt` turns wide into long, and `pivot` turns long into wide.

In [ ]:
wide = pd.DataFrame({"branch": ["Campus", "Mall"],
                     "Jan": [690, 410], "Feb": [655, 398], "Mar": [697, 500]})

long = wide.melt(id_vars="branch", var_name="month", value_name="revenue")
print(long)
print(long.pivot(index="branch", columns="month", values="revenue"))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Rows multiply | Merging on a key that is not unique in either table | Check `df[key].duplicated().sum()` before merging. |
| Rows vanish | Using the default inner join without noticing | Compare row counts; use `how="left"` to keep everything. |
| KeyError | Key columns with different names in the two tables | Use `left_on=` and `right_on=`, or rename first. |
| Mixed columns | Using `concat` on tables with different columns | Check that the column names match before stacking. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Product Names on Orders

Merge the orders with the products and show the five most ordered products by total quantity.

In [ ]:
# Write your code below

### In-Class Exercise 2: Spend per Segment

Join all three tables and calculate the total spent and the average order value for each customer segment.

In [ ]:
# Write your code below

### In-Class Exercise 3: Check the Join

Count how many orders would be lost by an inner join with the customers table, and list their order IDs.

In [ ]:
# Write your code below

### In-Class Exercise 4: Different Key Names

Rename `customer_id` to `client` in a copy of the customers table, then merge the orders with it using `left_on` and `right_on`. Check that the number of rows equals the inner join from the lecture.

In [ ]:
# Write your code below

## Summary

- Data is split into tables linked by key columns.
- `merge(on=key)` puts matching rows side by side.
- `how` decides what happens to rows without a match.
- `indicator=True` and `isin` find unmatched keys.
- `concat` stacks tables with the same columns.
- `melt` and `pivot` switch between long and wide.

## What's Next

Stage 2 is complete: you can load, clean, filter, group and join data. Level 2 begins with Stage 3, exploring data with charts, starting with Matplotlib.